Set environment

In [ ]:
import os
import warnings
from dotenv import load_dotenv
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
warnings.filterwarnings("ignore")
load_dotenv("./../.env")

True

Load Document

In [3]:
from pathlib import Path
dataset_path = Path("nvidia_dataset")
pdfs = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".pdf"):
            pdfs.append(os.path.join(root, file))
print(pdfs)


['nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf']


Transfer PDF into Langchain Document

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
docs = []
for pdf in pdfs:
    loader = PyMuPDFLoader(pdf)
    docs.extend(loader.load())

print(len(docs))


57


Split Documents into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=200, add_start_index = True)
chunks = text_splitter.split_documents(docs)
print(len(chunks))

247


Embed chunks into vectors

In [6]:
from langchain_ollama import OllamaEmbeddings
base_url = "http://localhost:11434"
model = "nomic-embed-text"
embeddings = OllamaEmbeddings(model=model, base_url=base_url)

Save vectors in a vector database

In [7]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

embedding_dim = len(embeddings.embed_query("Hello World"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index = index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

ids = vector_store.add_documents(documents=chunks)
print(ids)

['700783d6-bb99-4284-9a8d-bb28468ce339', 'b932ee8d-c1c7-47f2-baf6-10be8c8ec305', '69242a73-b951-42cb-b73f-5aeb53cf7098', 'f1f6c7ab-6d32-4ae8-8384-9d88ad432c21', 'ba6b5ea5-d7bc-4bea-9c91-d7ade6cce466', '13ca0e3e-9342-40d1-aa9e-693f8e5e5d1b', '3481767b-adf2-47aa-b4ba-ad58b35d2ef3', 'a0c86fdc-882a-4e32-a59a-7bf18e12832b', '754fe89a-42ca-4b01-88bb-d404a39e01b4', '6bcfaf09-6712-4786-af19-33ed4ff6aaf1', '9599fd44-1839-40fb-8f2b-d99d30019057', 'bbc09fb0-8b63-470e-a086-d892d49d474c', 'e4bc7860-74b3-4e76-9915-6e0389c0d1cb', '45995090-0b86-42a2-bbf5-2eb28b4343d6', 'dce6824f-329b-4da6-844b-4c89cee97e55', '9a1177d1-a3fc-4dbb-b201-4e874782e043', '1e10f9bf-3ba8-440a-bb5f-81602f637eba', 'f99fa0ba-04e8-4905-9bb1-4fd48de911ac', 'c433ecf5-1da3-4965-abec-52ac94f2eb59', 'eb36ae75-998d-4884-92b5-9945799fac58', '0e01aad0-538f-4104-9cc1-d7575ee42a78', 'dd0ef13b-b445-4a31-a3d0-5798695e9527', '7ec5c157-b246-44d7-933b-7183c3de4cba', 'cfd27f50-d3b4-4867-b52e-ecf9b8aff6f8', 'd6692bb1-8c51-41c5-9145-74c05a9d4d3e',

Save vector store

In [8]:
vector_store.index.ntotal

247

In [10]:
db_name = "nvidia_vector_store"
vector_store.save_local(db_name)

Search test

In [ ]:
test_question = "what is Stochastic Texture Filtering used for?"
test_result = vector_store.search(test_question, k=3, search_type="similarity")
print(test_result)

[Document(id='ebeaf4f5-6b3d-401f-b348-a31c4ab38579', metadata={'producer': 'Adobe PDF Library 25.1.250', 'creator': 'Acrobat PDFMaker 25 for Word', 'creationdate': '2025-05-12T15:44:14-04:00', 'source': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'file_path': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'total_pages': 57, 'format': 'PDF 1.4', 'title': '', 'author': 'NVIDIA', 'subject': '', 'keywords': '', 'moddate': '2025-05-12T15:45:50-04:00', 'trapped': '', 'modDate': "D:20250512154550-04'00'", 'creationDate': "D:20250512154414-04'00'", 'page': 40, 'start_index': 1669}, page_content='Stochastic Texture Filtering (STF) is used to introduce randomness into the texture sampling \nprocess to reduce visual artifacts like aliasing and moiré patterns when it is impractical to apply \ntraditional trilinear or anisotropic filtering, such as with Neural Texture Compression. In cases'), Document(id='700783d6-bb99-4284-9a8d-bb28468ce339', metadata={'producer': 'A

In [12]:
test_result

[Document(id='ebeaf4f5-6b3d-401f-b348-a31c4ab38579', metadata={'producer': 'Adobe PDF Library 25.1.250', 'creator': 'Acrobat PDFMaker 25 for Word', 'creationdate': '2025-05-12T15:44:14-04:00', 'source': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'file_path': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'total_pages': 57, 'format': 'PDF 1.4', 'title': '', 'author': 'NVIDIA', 'subject': '', 'keywords': '', 'moddate': '2025-05-12T15:45:50-04:00', 'trapped': '', 'modDate': "D:20250512154550-04'00'", 'creationDate': "D:20250512154414-04'00'", 'page': 40, 'start_index': 1669}, page_content='Stochastic Texture Filtering (STF) is used to introduce randomness into the texture sampling \nprocess to reduce visual artifacts like aliasing and moiré patterns when it is impractical to apply \ntraditional trilinear or anisotropic filtering, such as with Neural Texture Compression. In cases'),
 Document(id='700783d6-bb99-4284-9a8d-bb28468ce339', metadata={'producer': '

Load vector store from local

In [13]:
vector_store = FAISS.load_local(db_name, embeddings, allow_dangerous_deserialization=True)

Create a Retriever

In [14]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever.invoke(test_question)

[Document(id='ebeaf4f5-6b3d-401f-b348-a31c4ab38579', metadata={'producer': 'Adobe PDF Library 25.1.250', 'creator': 'Acrobat PDFMaker 25 for Word', 'creationdate': '2025-05-12T15:44:14-04:00', 'source': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'file_path': 'nvidia_dataset\\nvidia-rtx-blackwell-gpu-architecture.pdf', 'total_pages': 57, 'format': 'PDF 1.4', 'title': '', 'author': 'NVIDIA', 'subject': '', 'keywords': '', 'moddate': '2025-05-12T15:45:50-04:00', 'trapped': '', 'modDate': "D:20250512154550-04'00'", 'creationDate': "D:20250512154414-04'00'", 'page': 40, 'start_index': 1669}, page_content='Stochastic Texture Filtering (STF) is used to introduce randomness into the texture sampling \nprocess to reduce visual artifacts like aliasing and moiré patterns when it is impractical to apply \ntraditional trilinear or anisotropic filtering, such as with Neural Texture Compression. In cases'),
 Document(id='700783d6-bb99-4284-9a8d-bb28468ce339', metadata={'producer': '

Build chain

In [15]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [16]:
base_url = "http://localhost:11434"
model = "llama3.2"
chat = ChatOllama(model=model, base_url=base_url, temperature=0.9)

prompt = """
            You are a helpful assistant for question answering. Use the following context to answer the question. If you don't know the answer, say you don't know.
            Context: {context}
            Question: {question}
            Answer:
         """

prompt = ChatPromptTemplate.from_template(prompt)

In [19]:
from langchain_core.runnables import chain
@chain
def gen_context(chunks):
    context = ""
    for chunk in chunks:
        context += chunk.page_content + "\n\n"
    return context

print(gen_context.invoke(test_result))

Stochastic Texture Filtering (STF) is used to introduce randomness into the texture sampling 
process to reduce visual artifacts like aliasing and moiré patterns when it is impractical to apply 
traditional trilinear or anisotropic filtering, such as with Neural Texture Compression. In cases

V1.1 
 
 
 
 
 
NVIDIA RTX BLACKWELL  
GPU ARCHITECTURE 
Built for Neural Rendering

Neural Radiance Cache (NRC) ..................................................................................................................................... 42 
RTX Skin ................................................................................................................................................................................ 43 
RTX Neural Faces .............................................................................................................................................................. 44




In [20]:
rag_chain = {"question":RunnablePassthrough(), "context": retriever | gen_context} | prompt | chat | StrOutputParser()

In [ ]:
question = "what is Stochastic Texture Filtering used for?"
rag_chain.invoke(question)

'Stochastic Texture Filtering (STF) is used to introduce randomness into the texture sampling process to reduce visual artifacts like aliasing and moiré patterns.'